# A100 Smoke Benchmark Notes

This notebook records a lightweight RL-Kernel benchmark investigation on an A100 environment. The goal is to make the benchmark process reproducible and to validate whether the generated reports accurately describe the backend that was measured.

Scope:

- Record visible GPU hardware and runtime state.
- Run a minimal benchmark shape to validate the profiling path.
- Capture generated CSV/JSON reports.
- Compare benchmark status fields with runtime logs.
- Identify any reporting issue before using the data in benchmark documentation.

## Benchmark Method

This run uses a smoke-scale configuration. It is intended to validate benchmark wiring and report semantics, not to produce final performance claims.

Method:

- Select one visible A100 device with `CUDA_VISIBLE_DEVICES`.
- Use a small tensor shape to keep the run quick and easy to repeat.
- Use `warmup=0` and `repeat=1` for initial validation.
- Treat latency and throughput values as diagnostic observations only.
- Use larger shapes and repeated runs before publishing performance numbers.

In [1]:
# Check visible GPU state before selecting a device.
!nvidia-smi --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv,noheader,nounits

0, NVIDIA A100 80GB PCIe, 0, 81920, 0
1, NVIDIA A100 80GB PCIe, 0, 81920, 0
2, NVIDIA A100 80GB PCIe, 0, 81920, 0
3, NVIDIA A100 80GB PCIe, 0, 81920, 0
4, NVIDIA A100 80GB PCIe, 0, 81920, 0
5, NVIDIA A100 80GB PCIe, 0, 81920, 0
6, NVIDIA A100 80GB PCIe, 3, 81920, 0
7, NVIDIA A100 80GB PCIe, 0, 81920, 0


## Smoke Benchmark Command

The command below uses a minimal workload to exercise the benchmark runner, backend registry, CSV writer, and JSON writer. It is useful for checking whether the profiler records the correct backend status before running larger benchmark suites.

In [4]:
!CUDA_VISIBLE_DEVICES=2 python3 ../../scripts/run_profile_suite.py \
  --device cuda \
  --dtype float32 \
  --batch-sizes 2 \
  --seq-lens 16 \
  --vocab-sizes 1024 \
  --workloads logp-native,logp-fused,sampling-native \
  --warmup 0 \
  --repeat 1 \
  --output-dir . \
  --csv \
  --json

INFO 06-15 07:45:20 [RL-Kernel]: RL-Engine initialized with NVIDIA CUDA backend (Version: 12.9)
INFO 06-15 07:45:23 [RL-Kernel]: Profiling 'logp_native' | shape=(2,16,1024)
INFO 06-15 07:45:23 [RL-Kernel]: KernelRegistry initialized for cuda
WARNING 06-15 07:45:23 [RL-Kernel]: Core binary extension (_C) unavailable: cannot import name '_C' from 'rl_engine' (/mnt/cfs_bj_mt/workspace/limengjie03/my_school/RL-Kernel/rl_engine/__init__.py). Falling back to native code.
ERROR 06-15 07:45:23 [RL-Kernel]: Failed to instantiate CUDA_FUSED_LOGP_GENERIC: Base custom kernel 'fused_logp' is unavailable.
WARNING 06-15 07:45:23 [RL-Kernel]: Backend FLASHINFER unavailable: No module named 'rl_engine.kernels.ops.cuda.flashinfer'. Falling back...
WARNING 06-15 07:45:23 [RL-Kernel]: Backend TRITON_GENERIC unavailable: No module named 'rl_engine.kernels.ops.triton.generic'. Falling back...
INFO 06-15 07:45:23 [RL-Kernel]: Profiling 'logp_fused' | shape=(2,16,1024)
INFO 06-15 07:45:23 [RL-Kernel]: Profili

Equivalent command from the repository root:

```bash
CUDA_VISIBLE_DEVICES=2 python3 scripts/run_profile_suite.py \
  --device cuda \
  --dtype float32 \
  --batch-sizes 2 \
  --seq-lens 16 \
  --vocab-sizes 1024 \
  --workloads logp-native,logp-fused,sampling-native \
  --warmup 0 \
  --repeat 1 \
  --output-dir reports/a100-smoke \
  --csv \
  --json
```

Generated outputs include:

- `perf_report_NVIDIA_A100_80GB_PCIe.csv`
- `perf_report_NVIDIA_A100_80GB_PCIe_<timestamp>.json`

When the notebook is executed from `reports/a100-smoke/`, the command writes outputs to the current directory via `--output-dir .`.

In [5]:
from pathlib import Path
import pandas as pd

csv_path = Path('perf_report_NVIDIA_A100_80GB_PCIe.csv')
if not csv_path.exists():
    csv_path = Path('reports/a100-smoke/perf_report_NVIDIA_A100_80GB_PCIe.csv')

df = pd.read_csv(csv_path)
df[[
    'benchmark_name',
    'gpu_name',
    'gpu_architecture',
    'gpu_backend',
    'gpu_compute_capability',
    'batch_size',
    'seq_len',
    'vocab_size',
    'latency_ms',
    'tokens_per_sec',
    'peak_vram_gb',
    'status',
    'notes',
]]

,benchmark_name,gpu_name,gpu_architecture,gpu_backend,gpu_compute_capability,batch_size,seq_len,vocab_size,latency_ms,tokens_per_sec,peak_vram_gb,status,notes
0,logp_native,NVIDIA A100 80GB PCIe,Ampere,cuda,8.0,2,16,1024,703.949829,45.457785,0.000367,pass,NaN
1,logp_fused,NVIDIA A100 80GB PCIe,Ampere,cuda,8.0,2,16,1024,0.437248,73185.013122,0.000368,pass,NaN
2,sampling_native,NVIDIA A100 80GB PCIe,Ampere,cuda,8.0,2,16,1024,966.566895,33.106865,0.000073,pass,NaN


## Problem Summary and Proposed Fix

### Background

RL-Kernel's benchmark reports are intended to support hardware benchmark documentation and dashboard tables. For those reports to be trustworthy, each row must clearly distinguish between an optimized backend measurement and a fallback/native implementation measurement.

This notebook ran a smoke benchmark on an A100 environment with three workloads:

- `logp-native`
- `logp-fused`
- `sampling-native`

The run produced CSV and JSON reports successfully, which confirms that the benchmark orchestration path works.

### Evidence

The runtime log reported that optimized backends were unavailable:

```text
Core binary extension (_C) unavailable ... Falling back to native code.
Failed to instantiate CUDA_FUSED_LOGP_GENERIC: Base custom kernel 'fused_logp' is unavailable.
Backend FLASHINFER unavailable ... Falling back...
Backend TRITON_GENERIC unavailable ... Falling back...
```

However, the generated CSV still marked the `logp_fused` row as:

- `status = pass`
- `notes = NaN` / empty

This makes the report ambiguous: a reader may interpret `logp_fused` as a valid fused CUDA kernel measurement even though the logs indicate fallback behavior.

### Likely Root Cause

The `logp-fused` workload obtains the `logp` operator through the runtime registry and then calls `apply_fp32`. The benchmark path does not appear to verify that the selected operator is actually the fused CUDA backend. If the registry falls back to a native implementation, the benchmark still completes and is recorded as a successful `logp_fused` measurement.

### Proposed Fix

Before using these results in benchmark documentation, update the profiler so backend fallback is visible in machine-readable output.

Recommended implementation:

1. In `benchmarks/profiler.py`, make `_run_logp_fused_workload` verify that `kernel_registry.get_op("logp")` resolves to the expected fused CUDA backend.
2. If the fused backend is unavailable, emit a `blocked` metric with a clear `notes` value such as `fused CUDA backend unavailable; registry selected fallback backend`.
3. If fallback measurements are intentionally allowed later, record them under a separate benchmark name such as `logp_fallback` instead of `logp_fused`.
4. Add a regression test in `tests/test_profiler.py` to ensure fallback/native execution is never silently reported as `logp_fused` with `status=pass`.
5. After the reporting semantics are fixed, rerun A100 benchmarks with larger shapes and repeated measurements, then use those results in the benchmark dashboard.

Likely PR files:

- `benchmarks/profiler.py`
- `tests/test_profiler.py`
- `docs/benchmarking/README.md` after the profiler behavior is fixed

### Conclusion

The benchmark runner works, but current report semantics can overstate what was measured when optimized extensions are missing. The next contribution should fix benchmark reporting first; only then should A100 baseline data be promoted into documentation or dashboard tables.